In [ ]:
print("Setup: imports, landscapes, Touzet computation")

import numpy as np
import matplotlib.pyplot as plt
import mavenn
from pathlib import Path
from tqdm import tqdm
import sys
sys.path.insert(0, str(Path('../../src').resolve()))
sys.path.insert(0, str(Path('..').resolve()))
from landscape import Landscape
from touzet import tail_count
from cgf import psi_1, psi_2
import style_config as sc

TOUZET_KW = dict(tol=0.5, max_dict_size=10_000_000, max_time=15)


def standardize_theta(theta):
    theta = np.asarray(theta, dtype=float)
    centered = theta - theta.mean(axis=1, keepdims=True)
    scale = centered.std() * np.sqrt(theta.shape[0])
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Cannot standardize additive effects with zero variance.")
    return centered / scale


def random_sequence_sd(theta):
    return np.sqrt(np.mean(theta**2, axis=1).sum())


def run_touzet(theta, land, n_bins, F_lo, F_hi, cache_file, touzet_kw=TOUZET_KW):
    cache = Path(cache_file)
    if cache.exists():
        data = np.load(cache)
        print(f"  Loaded cache from {cache}")
        return data["bin_edges"], data["N_geq_lo"], data["N_geq_hi"]
    bin_edges = np.linspace(F_lo, F_hi, n_bins + 1)
    N_geq_lo = np.full(len(bin_edges), np.nan)
    N_geq_hi = np.full(len(bin_edges), np.nan)
    for i, F in enumerate(tqdm(bin_edges, desc=f"  Touzet")):
        try:
            result = tail_count(theta, F, **touzet_kw)
            if isinstance(result, tuple):
                N_geq_lo[i] = float(result[0])
                N_geq_hi[i] = float(result[1])
            else:
                N_geq_lo[i] = N_geq_hi[i] = float(result)
        except ValueError:
            print(f"  Timeout at bin {i}/{len(bin_edges)}, F={F:.3f}")
            break
    np.savez(cache, bin_edges=bin_edges, N_geq_lo=N_geq_lo, N_geq_hi=N_geq_hi)
    print(f"  Saved cache to {cache}")
    return bin_edges, N_geq_lo, N_geq_hi

print("\nPanel (a): Simulated, L=100, C=4 — Touzet")
L_a, C_a, seed_a = 100, 4, 42
rng_a = np.random.default_rng(seed_a)
theta_a = rng_a.standard_normal((L_a, C_a))
land_a = Landscape(theta_a)
mu_a = psi_1(theta_a, 0.0)
F_lo_a = mu_a + 0.7 * (land_a.F_max - mu_a)
print(f"  L={L_a}, C={C_a}, F_min={land_a.F_min:.3f}, F_max={land_a.F_max:.3f}, mu={mu_a:.3f}")
edges_a, Ngeq_lo_a, Ngeq_hi_a = run_touzet(theta_a, land_a, n_bins=40,
                                              F_lo=F_lo_a, F_hi=land_a.F_max,
                                              cache_file="touzet_sim_L100_C4.npz")

print("\nPanel (b): Simulated, L=100, C=20 — Touzet")
L_b, C_b, seed_b = 100, 20, 42
rng_b = np.random.default_rng(seed_b)
theta_b = rng_b.standard_normal((L_b, C_b))
land_b = Landscape(theta_b)
mu_b = psi_1(theta_b, 0.0)
F_lo_b = mu_b + 0.7 * (land_b.F_max - mu_b)
print(f"  L={L_b}, C={C_b}, F_min={land_b.F_min:.3f}, F_max={land_b.F_max:.3f}, mu={mu_b:.3f}")
edges_b, Ngeq_lo_b, Ngeq_hi_b = run_touzet(theta_b, land_b, n_bins=40,
                                              F_lo=F_lo_b, F_hi=land_b.F_max,
                                              cache_file="touzet_sim_L100_C20.npz")

print("\nPanel (c): lac promoter, L=75, C=4 — Touzet")
lac_model = mavenn.load_example_model('sortseq_mpa_additive')
theta_c = standardize_theta(lac_model.get_theta(gauge='consensus')['theta_lc'])
L_c, C_c = theta_c.shape
land_c = Landscape(theta_c)
mu_c = psi_1(theta_c, 0.0)
F_lo_c = mu_c + 0.7 * (land_c.F_max - mu_c)
print(f"  L={L_c}, C={C_c}, sd(F)={random_sequence_sd(theta_c):.3f}, F_min={land_c.F_min:.3f}, F_max={land_c.F_max:.3f}, mu={mu_c:.3f}")
edges_c, Ngeq_lo_c, Ngeq_hi_c = run_touzet(theta_c, land_c, n_bins=40,
                                              F_lo=F_lo_c, F_hi=land_c.F_max,
                                              cache_file="touzet_lac_zscore.npz")

print("\nPanel (d): GB1 protein, L=55, C=20 — Touzet")
gb1_model = mavenn.load_example_model('gb1_ge_additive')
theta_d = standardize_theta(gb1_model.get_theta(gauge='consensus')['theta_lc'])
L_d, C_d = theta_d.shape
land_d = Landscape(theta_d)
mu_d = psi_1(theta_d, 0.0)
F_lo_d = mu_d + 0.7 * (land_d.F_max - mu_d)
print(f"  L={L_d}, C={C_d}, sd(F)={random_sequence_sd(theta_d):.3f}, F_min={land_d.F_min:.3f}, F_max={land_d.F_max:.3f}, mu={mu_d:.3f}")
edges_d, Ngeq_lo_d, Ngeq_hi_d = run_touzet(theta_d, land_d, n_bins=40,
                                              F_lo=F_lo_d, F_hi=land_d.F_max,
                                              cache_file="touzet_gb1_zscore.npz")

print("\nSaving simulated theta matrices for use by other figures")
np.save("theta_sim_L100_C4.npy", theta_a)
np.save("theta_sim_L100_C20.npy", theta_b)
print("  Saved theta_sim_L100_C4.npy and theta_sim_L100_C20.npy")

print("\nDone with all computation.")

In [ ]:
print("Figure 2: log rho(F) — 2x2 layout")

def touzet_to_log_rho(bin_edges, N_geq):
    widths = bin_edges[1:] - bin_edges[:-1]
    diff = N_geq[:-1] - N_geq[1:]
    rho_est = diff / widths
    mask = np.isfinite(rho_est) & (rho_est > 0)
    log_rho = np.full_like(rho_est, np.nan)
    log_rho[mask] = np.log(rho_est[mask])
    return log_rho, mask

def plot_panel(ax, theta, land, L_val, C_val, log_rho_lo, log_rho_hi,
               bin_edges, mask_lo, mask_hi, show_legend=False, xlabel=r"$F$"):
    bounds_differ = np.any(np.isfinite(log_rho_lo) & np.isfinite(log_rho_hi) &
                           (np.abs(log_rho_hi - log_rho_lo) > 0.01))
    if bounds_differ:
        ax.stairs(log_rho_hi, bin_edges,
                  baseline=None, **sc.STYLES["bounds"])

    mu_0 = psi_1(theta, 0.0)
    F_fine = np.linspace(max(mu_0, land.F_min + 0.01), land.F_max - 0.01, 500)
    sigma_0 = np.sqrt(psi_2(theta, 0.0))
    CL = float(C_val ** L_val)
    rho_gauss = CL / (np.sqrt(2 * np.pi) * sigma_0) * np.exp(
        -(F_fine - mu_0)**2 / (2 * sigma_0**2))
    rho_saddle = land.density(F_fine, method="saddlepoint")
    valid_s = rho_saddle > 0

    ax.plot(F_fine[valid_s], np.log(rho_saddle[valid_s]), **sc.STYLES["saddle"])
    ax.plot(F_fine, np.log(rho_gauss), **sc.STYLES["bulk"])

    ax.stairs(log_rho_lo, bin_edges,
              baseline=None, **sc.STYLES["true"])

    ax.axvline(mu_0, **sc.STYLES["F_mean"])
    ax.axvline(land.F_max, **sc.STYLES["F_max"])
    ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel)

    if show_legend:
        ax.legend(**sc.LEGEND_KW, loc="lower left")

fig, axes = plt.subplots(2, 2, figsize=sc.FIGSIZE_2x2)
(ax_a, ax_b), (ax_c, ax_d) = axes

print("Panel (a): L=100, C=4 — Touzet")
log_rho_lo_a, mask_lo_a = touzet_to_log_rho(edges_a, Ngeq_lo_a)
log_rho_hi_a, mask_hi_a = touzet_to_log_rho(edges_a, Ngeq_hi_a)
plot_panel(ax_a, theta_a, land_a, L_a, C_a, log_rho_lo_a, log_rho_hi_a,
           edges_a, mask_lo_a, mask_hi_a, show_legend=True)
ax_a.set_ylabel(r"$\log\,\rho(F)$")
ax_a.set_title("(a)  Simulated, " + r"$L{=}100$, $C{=}4$", fontsize=sc.PANEL_TITLE_SIZE, loc="left")

print("Panel (b): L=100, C=20 — Touzet")
log_rho_lo_b, mask_lo_b = touzet_to_log_rho(edges_b, Ngeq_lo_b)
log_rho_hi_b, mask_hi_b = touzet_to_log_rho(edges_b, Ngeq_hi_b)
plot_panel(ax_b, theta_b, land_b, L_b, C_b, log_rho_lo_b, log_rho_hi_b,
           edges_b, mask_lo_b, mask_hi_b)
ax_b.set_ylabel(r"$\log\,\rho(F)$")
ax_b.set_title("(b)  Simulated, " + r"$L{=}100$, $C{=}20$", fontsize=sc.PANEL_TITLE_SIZE, loc="left")

print("Panel (c): lac promoter")
log_rho_lo_c, mask_lo_c = touzet_to_log_rho(edges_c, Ngeq_lo_c)
log_rho_hi_c, mask_hi_c = touzet_to_log_rho(edges_c, Ngeq_hi_c)
plot_panel(ax_c, theta_c, land_c, L_c, C_c, log_rho_lo_c, log_rho_hi_c,
           edges_c, mask_lo_c, mask_hi_c, xlabel=r"$F$ (z-score)")
ax_c.set_ylabel(r"$\log\,\rho(F)$")
ax_c.set_title("(c)  " + r"$\it{lac}$ promoter, $L{=}75$, $C{=}4$",
               fontsize=sc.PANEL_TITLE_SIZE, loc="left")

print("Panel (d): GB1 protein")
log_rho_lo_d, mask_lo_d = touzet_to_log_rho(edges_d, Ngeq_lo_d)
log_rho_hi_d, mask_hi_d = touzet_to_log_rho(edges_d, Ngeq_hi_d)
plot_panel(ax_d, theta_d, land_d, L_d, C_d, log_rho_lo_d, log_rho_hi_d,
           edges_d, mask_lo_d, mask_hi_d, xlabel=r"$F$ (z-score)")
ax_d.set_ylabel(r"$\log\,\rho(F)$")
ax_d.set_title("(d)  GB1 protein, " + r"$L{=}55$, $C{=}20$", fontsize=sc.PANEL_TITLE_SIZE, loc="left")

fig.tight_layout()
fig.savefig("fig2.pdf", bbox_inches="tight")
fig.savefig("fig2.png", dpi=300, bbox_inches="tight")
print("Saved fig2.pdf and fig2.png")
plt.show()